# 04 - Interaction Matrix

Builds the sparse playlist x track interaction matrix using the actual
project code (`src/features/interaction_matrix.py`), not a notebook-only
reimplementation. This is the input `LightFM` trains on in the next
notebook.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.features.interaction_matrix import InteractionMatrixBuilder

interactions_df = pd.read_csv('../data/processed/playlist_track_interactions.csv')
print(interactions_df.shape)
interactions_df.head()

(225776, 8)


,pid,playlist_name,track_uri,track_name,artist_name,album_name,popularity,track_genre
0,0,Acoustic Mix 0,spotify:track:5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,82,acoustic
1,0,Acoustic Mix 0,spotify:track:1EzrEOXmMH3G43AXT1y7pA,I'm Yours,Jason Mraz,We Sing. We Dance. We Steal Things.,80,acoustic
2,0,Acoustic Mix 0,spotify:track:08MFgEQeVLF37EyZ7jcwLc,Pano,Zack Tabudlo,Pano,75,acoustic
3,0,Acoustic Mix 0,spotify:track:0IktbUcnAGrvD03AWnz3Q8,Lucky,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,74,acoustic
4,0,Acoustic Mix 0,spotify:track:6Vc5wAMmXdKIAM7WUoEb7N,Say Something,A Great Big World;Christina Aguilera,Is There Anybody Out There?,74,acoustic


In [2]:
builder = InteractionMatrixBuilder()
coo = builder.build_coo_matrix(interactions_df)
print("Interaction matrix shape (playlists x tracks):", coo.shape)
print("Non-zero entries:", coo.nnz)
print(f"Density: {coo.nnz / (coo.shape[0] * coo.shape[1]):.6f}")

Interaction matrix shape (playlists x tracks): (674, 16804)
Non-zero entries: 225776
Density: 0.019934


## Sparsity is expected and healthy

A density this low (well under 1%) is completely normal for recommender interaction matrices - most users (playlists) never touch most items (tracks). This is exactly the setting collaborative filtering with implicit feedback is designed for.

In [3]:
interactions_per_playlist = np.asarray(coo.sum(axis=1)).flatten()
interactions_per_track = np.asarray(coo.sum(axis=0)).flatten()

print("Tracks per playlist -> mean: {:.1f}, min: {}, max: {}".format(
    interactions_per_playlist.mean(), interactions_per_playlist.min(), interactions_per_playlist.max()))
print("Playlists per track  -> mean: {:.2f}, min: {}, max: {}".format(
    interactions_per_track.mean(), interactions_per_track.min(), interactions_per_track.max()))

Tracks per playlist -> mean: 335.0, min: 284.0, max: 397.0
Playlists per track  -> mean: 13.44, min: 1.0, max: 16.0


In [4]:
from scipy import sparse

sparse.save_npz('../data/processed/interaction_matrix.npz', coo.tocsr())

import pickle
with open('../data/processed/id_mappings.pkl', 'wb') as f:
    pickle.dump({
        'playlist_to_idx': builder.playlist_to_idx,
        'track_to_idx': builder.track_to_idx,
        'idx_to_track': builder.idx_to_track,
    }, f)

print("Saved interaction_matrix.npz and id_mappings.pkl")

Saved interaction_matrix.npz and id_mappings.pkl
